# Reuse bots across multiple games

This is the notebook version of `sample_experiment.py`. Run the cells from top to bottom.
It creates one runner per entrant, reuses the same bots across five games, prints
each game's result, and then prints total points and games won per entrant.

Edit the configuration to choose your own bots, seeds, timeout, and game count.
Labels identify entrants; repeated classes still use separate bot instances.
Seats stay fixed. Each tied winner receives one win, so total wins may exceed
the number of games. Reusing runners retains lifetime bot memory;
`on_game_start()` lets each bot selectively reset per-game state.


## Find the project and import the API


In [1]:
import os
import sys
from pathlib import Path

# Find the repository whether the kernel starts in its root or experiments/.
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "src" / "harness.py").is_file()
        and (path / "src" / "bot_runner.py").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start this notebook inside the Megagem repository.")
os.chdir(PROJECT_ROOT)
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

In [2]:
from collections import Counter
from contextlib import ExitStack
from random import Random

from bot_runner import BotRunner
from bots import Bot
from custom_bots import SmallBidBot
from harness import play_game, print_result

## Configure the experiment

Use 3–5 entrants and unique labels. Both the game and bot seeds are explicit.


In [3]:
NUM_GAMES = 5
GAME_SEED = 42
BOT_SEED = 7
BOT_TIMEOUT = 1.0  # Use None to disable callback timeouts.

# Labels identify individual entrants, even when they use the same bot class.
PARTICIPANTS: tuple[tuple[str, type[Bot]], ...] = (
    ("SmallBidBot", SmallBidBot),
    ("DefaultBot A", Bot),
    ("DefaultBot B", Bot),
)

## Play games using persistent runners

The context manager closes workers even if execution raises an exception.
Rerunning this cell starts a fresh experiment with fresh runners and totals.


In [4]:
total_points: Counter[str] = Counter()
games_won: Counter[str] = Counter()

# Each runner is created once and reused in every game. ExitStack closes all
# workers at the end of the experiment, including when a game raises an error.
with ExitStack() as stack:
    runners = [
        stack.enter_context(
            BotRunner(bot_class, timeout=BOT_TIMEOUT, seed=BOT_SEED + index)
        )
        for index, (_, bot_class) in enumerate(PARTICIPANTS)
    ]
    for game_index in range(NUM_GAMES):
        print(f"\nGame {game_index + 1}/{NUM_GAMES}")
        print(
            "Seats: "
            + ", ".join(
                f"{index} = {label}" for index, (label, _) in enumerate(PARTICIPANTS)
            )
        )
        result = play_game(
            len(runners), bots=runners, rng=Random(GAME_SEED + game_index)
        )
        print_result(result)

        for player_id, score in enumerate(result.scores):
            total_points[PARTICIPANTS[player_id][0]] += score
        # Every tied winner receives one win; wins can sum to more than NUM_GAMES.
        for player_id in result.winner_ids:
            games_won[PARTICIPANTS[player_id][0]] += 1


Game 1/5
Seats: 0 = SmallBidBot, 1 = DefaultBot A, 2 = DefaultBot B


Game Over:
 Player 0: 237
 Player 1: 35
 Player 2: 35

 Player 0 wins with 237 coins!

Game 2/5
Seats: 0 = SmallBidBot, 1 = DefaultBot A, 2 = DefaultBot B


Game Over:
 Player 0: 231
 Player 1: 35
 Player 2: 35

 Player 0 wins with 231 coins!

Game 3/5
Seats: 0 = SmallBidBot, 1 = DefaultBot A, 2 = DefaultBot B
Game Over:
 Player 0: 236
 Player 1: 35
 Player 2: 35

 Player 0 wins with 236 coins!

Game 4/5
Seats: 0 = SmallBidBot, 1 = DefaultBot A, 2 = DefaultBot B
Game Over:
 Player 0: 236
 Player 1: 35
 Player 2: 35

 Player 0 wins with 236 coins!

Game 5/5
Seats: 0 = SmallBidBot, 1 = DefaultBot A, 2 = DefaultBot B
Game Over:
 Player 0: 215
 Player 1: 35
 Player 2: 35

 Player 0 wins with 215 coins!


## Print aggregate results

Counters return zero for an entrant that never won.


In [5]:
print(f"\nAggregate results across {NUM_GAMES} games")
print(f"{'Bot':<24} {'Total points':>12} {'Games won':>10}")
print("-" * 48)
for label, _ in PARTICIPANTS:
    print(f"{label:<24} {total_points[label]:>12} {games_won[label]:>10}")


Aggregate results across 5 games
Bot                      Total points  Games won
------------------------------------------------
SmallBidBot                      1155          5
DefaultBot A                      175          0
DefaultBot B                      175          0
